# Exploratory Data Analysis & Baseline Benchmarks
## AI Financial Coach for MFS Users (Sohoj)

This notebook performs comprehensive exploratory data analysis (EDA) across the 600-user primary synthetic cohort,
evaluating behavioral personas, spending composition, volatility, cultural seasonality, anomaly patterns, cold-start characteristics, 
and candidate feature correlations to establish empirical evidence for downstream ML model design.

In [ ]:
from pathlib import Path

import numpy as np

from ml.evaluation.evaluate_baselines import BaselineEvaluator
from ml.preprocessing.run_eda import EDAAnalyzer

# Initialize Analyzer
analyzer = EDAAnalyzer(data_dir=Path("data/exports"), figures_dir=Path("docs/data/figures/eda"))
monthly_df = analyzer.compute_monthly_aggregations()
print(
    f"Loaded {len(analyzer.df_users)} users and {len(analyzer.df_txns):,} transactions across 12 months."
)

### 1. Behavioral Persona Profiles & Spending Divergence
Examines the six behavioral archetypes and drifting personas across savings rates and cash-out ratios.

In [ ]:
persona_summary = (
    monthly_df.groupby("true_persona")
    .agg(
        mean_savings_rate=("savings_rate", "mean"),
        mean_cashout_ratio=("cashout_ratio", "mean"),
        mean_discretionary_share=("discretionary_share", "mean"),
        mean_necessity_share=("necessity_share", "mean"),
    )
    .reset_index()
)
persona_summary

### 2. Spending Composition & Engel's Law Elasticity
Evaluates expenditure share across income quintiles, validating negative correlation with necessity spend.

In [ ]:
user_annual = (
    monthly_df.groupby("user_id")
    .agg(
        baseline_income=("baseline_income", "first"),
        total_nec=("necessity_spend", "sum"),
        total_disc=("discretionary_spend", "sum"),
    )
    .reset_index()
)
user_annual["necessity_share"] = user_annual["total_nec"] / (
    user_annual["total_nec"] + user_annual["total_disc"]
)
corr = user_annual["baseline_income"].corr(user_annual["necessity_share"])
print(f"Engel's Law Pearson correlation r = {corr:.2f}")

### 3. Volatility & Income Instability by Occupation
Compares income and expenditure coefficients of variation (CV) across occupational segments.

In [ ]:
user_vol = (
    monthly_df.groupby("user_id")
    .agg(
        occupation=("true_occupation", "first"),
        income_std=("monthly_inflow", "std"),
        income_mean=("monthly_inflow", "mean"),
    )
    .reset_index()
)
user_vol["income_cv"] = user_vol["income_std"] / np.maximum(100.0, user_vol["income_mean"])
user_vol.groupby("occupation")["income_cv"].describe()

### 4. Cultural Calendar & Seasonality Dynamics (Dual Eid)
Tracks transaction outflows across months, capturing March (Eid-ul-Fitr) and May (Eid-ul-Adha) surges.

In [ ]:
monthly_vol = (
    analyzer.df_txns.groupby(["month", "txn_type"])["amount"].sum().unstack(fill_value=0.0)
)
monthly_vol / 1e6  # Volume in Millions BDT

### 5. Ground-Truth Anomaly Distributions & Severity
Inspects the 7,800 injected anomalies across anomaly categories and severity distributions.

In [ ]:
analyzer.df_txn_gt["anomaly_type"].value_counts(normalize=True) * 100

### 6. Cold-Start Analysis & Category Sparsity
Audits data sufficiency in initial user months and verifies the requirement for peer-group fallbacks.

In [ ]:
user_cat_counts = (
    analyzer.df_txns[analyzer.df_txns["txn_type"] == "expense"]
    .groupby(["user_id", "category"])
    .size()
)
under_10_pct = (user_cat_counts < 10).mean() * 100
print(f"Share of (user, category) pairs with fewer than 10 transactions: {under_10_pct:.1f}%")

### 7. Correlation Heatmap & Multicollinearity
Evaluates correlation matrix across candidate rolling features to guide feature selection.

In [ ]:
feature_panel = monthly_df[
    [
        "savings_rate",
        "necessity_share",
        "discretionary_share",
        "cashout_ratio",
        "cashout_count",
        "txn_count",
        "avg_txn_amount",
        "monthly_inflow",
        "monthly_outflow",
    ]
]
feature_panel.corr()

### 8. Rule-Based Baselines & Benchmark Evaluation
Runs deterministic baseline models across validation users and prints benchmark performance.

In [ ]:
evaluator = BaselineEvaluator(data_dir=Path("data/exports"))
report = evaluator.run_all_evaluations()
print(f"Classifier Macro F1: {report.classifier_macro_f1:.4f}")
print(f"Anomaly Detector F1: {report.anomaly_f1:.4f} (ROC-AUC: {report.anomaly_roc_auc:.4f})")
print(f"Forecaster 3M SMA sMAPE: {report.sma_smape:.2f}%")